In [21]:
# @title
# ============================================================
# 0014 — VOILÀ APPLICATION
# CELL 0 — RUNTIME CONFIGURATION
# ============================================================

from pathlib import Path
import json
import html

import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "009_final_results"
)
print("FINAL DIR:")
print(FINAL_DIR)

print("\nFILES:")
if FINAL_DIR.exists():
    for f in FINAL_DIR.iterdir():
        print(f.name)
else:
    print("DIRECTORY DOES NOT EXIST")

MODEL_PATH = FINAL_DIR / "random_forest_final.pkl"
PREDICTIONS_PATH = FINAL_DIR / "prediction_results.csv"
MODELING_PATH = FINAL_DIR / "fitbit_modeling_features.csv"
METADATA_PATH = FINAL_DIR / "model_metadata.json"


# ------------------------------------------------------------
# UI CONFIGURATION
# ------------------------------------------------------------

HISTORY_DAYS = 30
SHAP_TOP_N = 5


# ------------------------------------------------------------
# CHECK FINAL ARTIFACTS + DEBUG
# ------------------------------------------------------------

print("=" * 60)
print("CHECKING FINAL ARTIFACTS")
print("=" * 60)

print("\nFINAL_DIR:")
print(FINAL_DIR)

print("\nDIRECTORY EXISTS:")
print(FINAL_DIR.exists())


if FINAL_DIR.exists():

    print("\nFILES FOUND:")

    for f in FINAL_DIR.iterdir():
        print(" -", f.name)

else:

    print("\nFINAL DIRECTORY NOT FOUND")


required_files = {
    "Random Forest model": MODEL_PATH,
    "Prediction results": PREDICTIONS_PATH,
    "Modeling dataset": MODELING_PATH,
    "Model metadata": METADATA_PATH,
}


missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Не знайдено необхідні фінальні артефакти:\n\n"
        + "\n".join(missing_files)
    )


print("\n✓ All final artifacts found")


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

modeling = pd.read_csv(
    MODELING_PATH
)

modeling["ActivityDate"] = pd.to_datetime(
    modeling["ActivityDate"]
)

modeling = (
    modeling
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)


predictions_df = pd.read_csv(
    PREDICTIONS_PATH
)

predictions_df["ActivityDate"] = pd.to_datetime(
    predictions_df["ActivityDate"]
)

predictions_df = (
    predictions_df
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# LOAD METADATA
# ------------------------------------------------------------

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    model_metadata = json.load(f)


model_name = model_metadata.get(
    "model",
    "Random Forest — All Features"
)

target_name = model_metadata.get(
    "target",
    "target_calories_next_day"
)


# ------------------------------------------------------------
# MODEL FEATURES
# ------------------------------------------------------------

feature_columns = list(
    model_metadata.get(
        "features",
        []
    )
)


# ------------------------------------------------------------
# DEMO / VALIDATION INDEX
# ------------------------------------------------------------

demo_keys = (
    predictions_df[
        ["Id", "ActivityDate"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)

user_ids = sorted(
    demo_keys["Id"].unique()
)


# ------------------------------------------------------------
# LAZY MODEL STATE
# ------------------------------------------------------------

rf_model = None
shap_explainer = None
SHAP_AVAILABLE = None


# ------------------------------------------------------------
# LAZY MODEL LOADER
# ------------------------------------------------------------

def get_model():
    global rf_model
    global feature_columns

    if rf_model is None:

        rf_model = joblib.load(
            MODEL_PATH
        )

        if not feature_columns:
            feature_columns = list(
                rf_model.feature_names_in_
            )

        if len(feature_columns) != 65:
            raise ValueError(
                "Фінальна модель повинна мати "
                f"65 ознак, отримано "
                f"{len(feature_columns)}."
            )

    return rf_model


# ------------------------------------------------------------
# LAZY SHAP LOADER
# ------------------------------------------------------------

def get_shap_explainer():

    global shap_explainer
    global SHAP_AVAILABLE

    if SHAP_AVAILABLE is False:
        return None

    if shap_explainer is None:

        try:

            import shap

            model = get_model()

            shap_explainer = (
                shap.TreeExplainer(model)
            )

            SHAP_AVAILABLE = True

        except Exception:

            SHAP_AVAILABLE = False
            shap_explainer = None

    return shap_explainer


# ------------------------------------------------------------
# TEXT SAFETY
# ------------------------------------------------------------

def safe_text(value):

    return html.escape(
        str(value)
    )

FINAL DIR:
/content/drive/MyDrive/FitnessML_Master/results/009_final_results

FILES:
final_model_comparison.csv
final_candidate_models.csv
009_final_model_comparison.xlsx
prediction_results.csv
fitbit_modeling_features.csv
random_forest_final.pkl
model_metadata.json
ui_config.json
CHECKING FINAL ARTIFACTS

FINAL_DIR:
/content/drive/MyDrive/FitnessML_Master/results/009_final_results

DIRECTORY EXISTS:
True

FILES FOUND:
 - final_model_comparison.csv
 - final_candidate_models.csv
 - 009_final_model_comparison.xlsx
 - prediction_results.csv
 - fitbit_modeling_features.csv
 - random_forest_final.pkl
 - model_metadata.json
 - ui_config.json

✓ All final artifacts found


In [22]:
# @title
# ============================================================
# CELL 1 — DATA AND PREDICTION ENGINE
# ============================================================


# ------------------------------------------------------------
# DATE / DATA ACCESS
# ------------------------------------------------------------

def get_dates_for_user(user_id):
    """
    Повертає доступні дати для вибраного користувача.
    """

    return (
        demo_keys.loc[
            demo_keys["Id"] == user_id,
            "ActivityDate"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


def get_modeling_row(
    user_id,
    activity_date
):
    """
    Повертає один рядок modeling dataset
    для конкретного користувача та дати.
    """

    activity_date = pd.Timestamp(
        activity_date
    )

    rows = modeling.loc[
        (modeling["Id"] == user_id)
        &
        (
            modeling["ActivityDate"]
            == activity_date
        )
    ]

    if rows.empty:
        raise ValueError(
            "Не знайдено відповідний рядок "
            "у modeling dataset."
        )

    return rows.iloc[0]


def get_prediction_row(
    user_id,
    activity_date
):
    """
    Повертає рядок із результатами підготовленого
    prediction dataset.
    """

    activity_date = pd.Timestamp(
        activity_date
    )

    rows = predictions_df.loc[
        (predictions_df["Id"] == user_id)
        &
        (
            predictions_df["ActivityDate"]
            == activity_date
        )
    ]

    if rows.empty:
        raise ValueError(
            "Не знайдено відповідний рядок "
            "у prediction results."
        )

    return rows.iloc[0]


# ------------------------------------------------------------
# USER HISTORY
# ------------------------------------------------------------

def get_user_history(
    user_id,
    activity_date,
    days=30
):
    """
    Повертає історію активності користувача
    до вибраної дати включно.
    """

    activity_date = pd.Timestamp(
        activity_date
    )

    return (
        modeling.loc[
            (modeling["Id"] == user_id)
            &
            (
                modeling["ActivityDate"]
                <= activity_date
            )
        ]
        .sort_values("ActivityDate")
        .tail(days)
        .copy()
    )


def get_recent_calories(
    user_id,
    activity_date,
    days=7
):
    """
    Повертає Calories за останні N доступних днів.
    """

    history = get_user_history(
        user_id,
        activity_date,
        days=days
    )

    if history.empty:
        return pd.Series(
            dtype=float
        )

    return history["Calories"]


# ------------------------------------------------------------
# BASELINES
# ------------------------------------------------------------

def baseline_7day_mean(
    user_id,
    activity_date
):
    """
    7-day mean baseline.
    """

    calories = get_recent_calories(
        user_id,
        activity_date,
        days=7
    )

    if calories.empty:
        return np.nan

    return float(
        calories.mean()
    )


def baseline_user_mean(
    user_id,
    activity_date
):
    """
    Середнє Calories конкретного користувача
    за доступною історією.
    """

    history = get_user_history(
        user_id,
        activity_date,
        days=len(
            modeling.loc[
                modeling["Id"] == user_id
            ]
        )
    )

    if history.empty:
        return np.nan

    return float(
        history["Calories"].mean()
    )


# ------------------------------------------------------------
# FEATURE VECTOR
# ------------------------------------------------------------

def make_feature_vector(
    modeling_row
):
    """
    Формує DataFrame з 65 ознаками
    у правильному порядку для Random Forest.
    """

    model = get_model()

    missing = [
        feature
        for feature in feature_columns
        if feature not in modeling_row.index
    ]

    if missing:
        raise ValueError(
            "У рядку відсутні ознаки:\n"
            + "\n".join(missing)
        )

    X = pd.DataFrame(
        [[
            modeling_row[feature]
            for feature in feature_columns
        ]],
        columns=feature_columns
    )

    if X.shape[1] != 65:
        raise ValueError(
            "Некоректна кількість ознак: "
            f"{X.shape[1]}. Очікується 65."
        )

    return X


# ------------------------------------------------------------
# PREDICTION
# ------------------------------------------------------------

def predict_calories(
    modeling_row
):
    """
    Формує прогноз наступного дня.
    """

    model = get_model()

    X = make_feature_vector(
        modeling_row
    )

    prediction = model.predict(X)

    return float(
        prediction[0]
    )


# ------------------------------------------------------------
# COMPLETE PREDICTION RESULT
# ------------------------------------------------------------

def build_prediction_result(
    user_id,
    activity_date,
    validation=False
):
    """
    Формує єдиний словник результату,
    який використовується UI.
    """

    modeling_row = get_modeling_row(
        user_id,
        activity_date
    )

    prediction_row = get_prediction_row(
        user_id,
        activity_date
    )

    predicted = predict_calories(
        modeling_row
    )

    baseline_7day = baseline_7day_mean(
        user_id,
        activity_date
    )

    baseline_user = baseline_user_mean(
        user_id,
        activity_date
    )

    current_calories = float(
        modeling_row["Calories"]
    )

    result = {
        "user_id": user_id,
        "activity_date": pd.Timestamp(
            activity_date
        ),
        "predicted": predicted,
        "baseline_7day": baseline_7day,
        "baseline_user": baseline_user,
        "current_calories": current_calories,
        "modeling_row": modeling_row,
    }

    # --------------------------------------------------------
    # VALIDATION INFORMATION
    # --------------------------------------------------------

    if validation:

        actual = float(
            prediction_row["ActualCalories"]
        )

        result["actual"] = actual

        result["absolute_error"] = abs(
            predicted - actual
        )

        result["signed_error"] = (
            predicted - actual
        )

    return result

In [24]:
# @title
# ============================================================
# CELL 2 — UI STYLE SYSTEM
# ============================================================

UI_STYLE_CSS = """
<style>

:root {
    --ui-border: #d9dde3;
    --ui-background: #ffffff;
    --ui-soft: #f4f5f7;
    --ui-text: #20242a;
    --ui-muted: #68707a;

    --ui-success-bg: #edf7ef;
    --ui-success-border: #b9d9bf;

    --ui-warning-bg: #fff8e8;
    --ui-warning-border: #ead39a;

    --ui-error-bg: #fff1f1;
    --ui-error-border: #e2bcbc;
}


/* --------------------------------------------------------
   PAGE
   -------------------------------------------------------- */

.ui-page {
    max-width: 1180px;
    margin: 20px auto 40px auto;
    font-family: Arial, sans-serif;
    color: var(--ui-text);
}


/* --------------------------------------------------------
   HEADER
   -------------------------------------------------------- */

.ui-header {
    padding: 28px 32px;
    margin-bottom: 16px;
    background: var(--ui-background);
    border: 1px solid var(--ui-border);
    border-radius: 18px;
    box-shadow: 0 5px 20px rgba(0, 0, 0, 0.07);
}

.ui-title {
    font-size: 30px;
    font-weight: 700;
    line-height: 1.2;
    margin-bottom: 7px;
}

.ui-subtitle {
    font-size: 15px;
    color: var(--ui-muted);
    line-height: 1.5;
}


/* --------------------------------------------------------
   SECTION
   -------------------------------------------------------- */

.ui-section {
    margin-bottom: 16px;
}

.ui-section-title {
    font-size: 19px;
    font-weight: 700;
    margin-bottom: 13px;
}

.ui-section-subtitle {
    font-size: 13px;
    color: var(--ui-muted);
    margin-top: -7px;
    margin-bottom: 13px;
}


/* --------------------------------------------------------
   CARD
   -------------------------------------------------------- */

.ui-card {
    padding: 20px;
    margin-bottom: 16px;
    background: var(--ui-background);
    border: 1px solid var(--ui-border);
    border-radius: 15px;
    box-shadow: 0 3px 14px rgba(0, 0, 0, 0.05);
}

.ui-card-title {
    font-size: 18px;
    font-weight: 700;
    margin-bottom: 13px;
}

.ui-card-note {
    margin-top: 10px;
    font-size: 13px;
    line-height: 1.5;
    color: var(--ui-muted);
}


/* --------------------------------------------------------
   METRIC CARDS
   -------------------------------------------------------- */

.ui-metric-grid {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
}

.ui-metric {
    flex: 1 1 180px;
    min-width: 175px;
    padding: 17px;
    background: var(--ui-soft);
    border-radius: 12px;
}

.ui-metric-label {
    font-size: 12px;
    color: var(--ui-muted);
    margin-bottom: 7px;
}

.ui-metric-value {
    font-size: 23px;
    font-weight: 700;
    line-height: 1.2;
}

.ui-metric-small {
    font-size: 13px;
    color: var(--ui-muted);
    margin-top: 5px;
}


/* --------------------------------------------------------
   FORECAST
   -------------------------------------------------------- */

.ui-forecast {
    padding: 25px;
    border-radius: 15px;
    background: var(--ui-soft);
    border: 1px solid var(--ui-border);
    margin-bottom: 15px;
}

.ui-forecast-label {
    font-size: 13px;
    color: var(--ui-muted);
    margin-bottom: 5px;
}

.ui-forecast-value {
    font-size: 38px;
    font-weight: 700;
    line-height: 1.1;
}

.ui-forecast-unit {
    font-size: 15px;
    font-weight: 400;
    color: var(--ui-muted);
}


/* --------------------------------------------------------
   MODE BADGE
   -------------------------------------------------------- */

.ui-mode {
    display: inline-block;
    padding: 6px 10px;
    border-radius: 8px;
    font-size: 12px;
    font-weight: 600;
    background: var(--ui-soft);
    border: 1px solid var(--ui-border);
}


/* --------------------------------------------------------
   STATUS
   -------------------------------------------------------- */

.ui-status {
    padding: 13px 15px;
    border-radius: 10px;
    background: var(--ui-soft);
    border: 1px solid var(--ui-border);
    font-size: 13px;
    line-height: 1.5;
}

.ui-status-success {
    background: var(--ui-success-bg);
    border-color: var(--ui-success-border);
}

.ui-status-warning {
    background: var(--ui-warning-bg);
    border-color: var(--ui-warning-border);
}

.ui-status-error {
    background: var(--ui-error-bg);
    border-color: var(--ui-error-border);
}


/* --------------------------------------------------------
   SHAP FACTOR
   -------------------------------------------------------- */

.ui-factor {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 11px 13px;
    margin-bottom: 7px;
    border-radius: 9px;
    background: var(--ui-soft);
}

.ui-factor-name {
    font-size: 13px;
    font-weight: 600;
    overflow-wrap: anywhere;
}

.ui-factor-value {
    margin-left: 15px;
    font-size: 13px;
    font-weight: 700;
    white-space: nowrap;
}


/* --------------------------------------------------------
   TWO COLUMN LAYOUT
   -------------------------------------------------------- */

.ui-two-columns {
    display: flex;
    flex-wrap: wrap;
    gap: 16px;
}

.ui-column {
    flex: 1 1 480px;
}


/* --------------------------------------------------------
   TABLE
   -------------------------------------------------------- */

.ui-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}

.ui-table th {
    text-align: left;
    padding: 10px;
    background: var(--ui-soft);
    border-bottom: 1px solid var(--ui-border);
}

.ui-table td {
    padding: 10px;
    border-bottom: 1px solid var(--ui-border);
}


/* --------------------------------------------------------
   ERROR
   -------------------------------------------------------- */

.ui-error {
    padding: 15px;
    border-radius: 10px;
    background: var(--ui-error-bg);
    border: 1px solid var(--ui-error-border);
    color: #7f3030;
    font-size: 13px;
    line-height: 1.5;
}


/* --------------------------------------------------------
   FOOTNOTE
   -------------------------------------------------------- */

.ui-footnote {
    margin-top: 8px;
    font-size: 12px;
    color: var(--ui-muted);
    line-height: 1.5;
}


/* --------------------------------------------------------
   RESPONSIVE
   -------------------------------------------------------- */

@media (max-width: 700px) {

    .ui-page {
        margin: 10px;
    }

    .ui-title {
        font-size: 24px;
    }

    .ui-forecast-value {
        font-size: 32px;
    }

    .ui-metric {
        min-width: 140px;
    }

}

</style>
"""


# ------------------------------------------------------------
# INSERT CSS AS IPYWIDGET
# ------------------------------------------------------------

ui_style_widget = widgets.HTML(
    value=UI_STYLE_CSS
)


print("✓ UI style system loaded")

✓ UI style system loaded


In [25]:
# @title
# ------------------------------------------------------------
# WIDGET ENGINE TEST
# ------------------------------------------------------------

widget_test = widgets.HTML(
    value="""
    <div style="
        padding:15px;
        border:1px solid #888;
        border-radius:10px;
    ">
    <b>Widget engine alive</b>
    </div>
    """
)

display(widget_test)

print("✓ Widget display test passed")

HTML(value='\n    <div style="\n        padding:15px;\n        border:1px solid #888;\n        border-radius:1…

✓ Widget display test passed


In [26]:
# @title
# ============================================================
# CELL 3 — DASHBOARD CONTROLS
# ============================================================

# ------------------------------------------------------------
# USER SELECTOR
# ------------------------------------------------------------

user_dropdown = widgets.Dropdown(
    options=user_ids,
    value=user_ids[0],
    description="Користувач:",
    layout=widgets.Layout(
        width="320px"
    ),
    style={
        "description_width": "90px"
    }
)


# ------------------------------------------------------------
# MODE SELECTOR
# ------------------------------------------------------------

mode_dropdown = widgets.Dropdown(
    options=[
        ("Реальний прогноз", "real"),
        ("Валідація", "validation"),
        ("Демо", "demo"),
    ],
    value="real",
    description="Режим:",
    layout=widgets.Layout(
        width="320px"
    ),
    style={
        "description_width": "90px"
    },
)


# ------------------------------------------------------------
# DATE SELECTOR
# ------------------------------------------------------------

date_dropdown = widgets.Dropdown(
    options=[],
    description="Дата:",
    layout=widgets.Layout(
        width="320px"
    ),
    style={
        "description_width": "90px"
    },
)


# ------------------------------------------------------------
# FORECAST BUTTON
# ------------------------------------------------------------

forecast_button = widgets.Button(
    description="Сформувати прогноз",
    button_style="primary",
    icon="line-chart",
    tooltip="Сформувати прогноз енергетичних витрат на наступний день.",
    layout=widgets.Layout(
        width="220px",
        height="40px",
        margin="8px 0 0 0",
    ),
)


# ------------------------------------------------------------
# DASHBOARD OUTPUT
# ------------------------------------------------------------

dashboard_output = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)


# ------------------------------------------------------------
# UPDATE DATE OPTIONS
# ------------------------------------------------------------

def update_date_options(change=None):
    """
    Оновлює доступні дати після зміни користувача.
    """

    selected_user = user_dropdown.value

    if selected_user is None:
        date_dropdown.options = []
        date_dropdown.value = None
        return

    try:

        dates = get_dates_for_user(
            selected_user
        )

        date_options = [
            (
                pd.Timestamp(date).strftime("%Y-%m-%d"),
                pd.Timestamp(date)
            )
            for date in dates
        ]

        date_dropdown.options = date_options

        if date_options:
            date_dropdown.value = date_options[-1][1]

        else:
            date_dropdown.value = None

    except Exception:

        date_dropdown.options = []
        date_dropdown.value = None


# ------------------------------------------------------------
# MODE CHANGE
# ------------------------------------------------------------

def on_mode_change(change):
    """
    Оновлює підказку кнопки залежно від режиму.
    """

    if change["name"] != "value":
        return

    if change["new"] == "real":

        forecast_button.tooltip = (
            "Сформувати реальний прогноз "
            "наступного дня без відображення "
            "відомого фактичного значення."
        )

    elif change["new"] == "validation":

        forecast_button.tooltip = (
            "Сформувати прогноз та порівняти "
            "його з відомим фактичним значенням."
        )

    else:

        forecast_button.tooltip = (
            "Демонстраційний режим роботи "
            "системи прогнозування."
        )


# ------------------------------------------------------------
# CALLBACKS
# ------------------------------------------------------------

user_dropdown.observe(
    update_date_options,
    names="value"
)

mode_dropdown.observe(
    on_mode_change,
    names="value"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

update_date_options()

print("✓ Dashboard controls initialized")

✓ Dashboard controls initialized


In [28]:
# @title
# ============================================================
# CELL 4 — FORECAST RENDERER
# ============================================================


# ------------------------------------------------------------
# FORMATTING HELPERS
# ------------------------------------------------------------

def format_kcal(value):
    """
    Форматування значення енергетичних витрат.
    """

    if value is None or pd.isna(value):
        return "—"

    return (
        f"{float(value):,.0f}"
        .replace(",", " ")
        + " ккал"
    )


def format_signed_kcal(value):
    """
    Форматування signed-різниці.
    """

    if value is None or pd.isna(value):
        return "—"

    value = float(value)

    sign = "+" if value > 0 else ""

    return (
        f"{sign}{value:,.0f}"
        .replace(",", " ")
        + " ккал"
    )


def format_percent(value):
    """
    Форматування відсоткового значення.
    """

    if value is None or pd.isna(value):
        return "—"

    return f"{float(value):.1f}%"


# ------------------------------------------------------------
# FORECAST CARD
# ------------------------------------------------------------

def render_forecast_card(result):
    """
    Основна картка прогнозу.
    """

    predicted = result.get(
        "predicted",
        np.nan
    )

    current = result.get(
        "current_calories",
        np.nan
    )

    activity_date = result.get(
        "activity_date"
    )

    if pd.isna(predicted):

        return widgets.HTML(
            value="""
            <div class="ui-error">
                Не вдалося сформувати прогноз.
            </div>
            """
        )

    if activity_date is not None:

        next_date = (
            pd.Timestamp(activity_date)
            + pd.Timedelta(days=1)
        )

        next_date_text = (
            next_date.strftime("%d.%m.%Y")
        )

    else:

        next_date_text = "на наступний день"

    current_text = format_kcal(
        current
    )

    predicted_text = format_kcal(
        predicted
    )

    html = f"""
    <div class="ui-forecast">

        <div class="ui-forecast-label">
            Прогноз добових енергетичних витрат
            на {next_date_text}
        </div>

        <div class="ui-forecast-value">
            {predicted_text}
        </div>

        <div class="ui-card-note">
            Поточне значення Calories:
            <b>{current_text}</b>
        </div>

    </div>
    """

    return widgets.HTML(
        value=html
    )


# ------------------------------------------------------------
# FORECAST STATUS
# ------------------------------------------------------------

def render_forecast_status(
    result,
    mode="real"
):
    """
    Статус результату прогнозування.
    """

    if result is None:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                Система готова до прогнозування.
            </div>
            """
        )

    predicted = result.get(
        "predicted",
        np.nan
    )

    if pd.isna(predicted):

        return widgets.HTML(
            value="""
            <div class="ui-status ui-status-error">
                Прогноз не сформовано.
            </div>
            """
        )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    if mode == "validation":

        actual = result.get(
            "actual",
            np.nan
        )

        error = result.get(
            "absolute_error",
            np.nan
        )

        if not pd.isna(actual):

            return widgets.HTML(
                value=f"""
                <div class="ui-status ui-status-success">

                    <b>Валідацію виконано.</b>

                    Фактичне значення:
                    <b>{format_kcal(actual)}</b>

                    <br>

                    Абсолютна похибка:
                    <b>{format_kcal(error)}</b>

                </div>
                """
            )

    # --------------------------------------------------------
    # DEMO
    # --------------------------------------------------------

    if mode == "demo":

        return widgets.HTML(
            value="""
            <div class="ui-status ui-status-warning">

                <b>Демо-режим.</b>

                Результат призначений для
                демонстрації роботи інтерфейсу.

            </div>
            """
        )

    # --------------------------------------------------------
    # REAL PREDICTION
    # --------------------------------------------------------

    return widgets.HTML(
        value="""
        <div class="ui-status ui-status-success">

            <b>Прогноз успішно сформовано.</b>

            Модель Random Forest використала
            доступні часові та активнісні характеристики.

        </div>
        """
    )


print("✓ Forecast renderer loaded")

✓ Forecast renderer loaded


In [29]:
# @title
# ============================================================
# CELL 5 — USER HISTORY
# ============================================================


# ------------------------------------------------------------
# PREPARE HISTORY DATA
# ------------------------------------------------------------

def prepare_history_data(
    user_id,
    activity_date,
    days=30
):
    """
    Підготовка історичних даних користувача
    для відображення у dashboard.
    """

    history = get_user_history(
        user_id,
        activity_date,
        days=days
    )

    if history.empty:

        return pd.DataFrame(
            columns=[
                "ActivityDate",
                "Calories"
            ]
        )

    history = history[
        [
            "ActivityDate",
            "Calories"
        ]
    ].copy()

    history["ActivityDate"] = pd.to_datetime(
        history["ActivityDate"]
    )

    history["Calories"] = pd.to_numeric(
        history["Calories"],
        errors="coerce"
    )

    return history.dropna(
        subset=["Calories"]
    )


# ------------------------------------------------------------
# HISTORY CHART
# ------------------------------------------------------------

def render_history_chart(
    user_id,
    activity_date,
    days=30
):
    """
    Побудова графіка історії Calories.
    """

    history = prepare_history_data(
        user_id,
        activity_date,
        days=days
    )

    if history.empty:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                Історичні дані для цього користувача
                відсутні.
            </div>
            """
        )

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(
        figsize=(10, 3.8)
    )

    ax.plot(
        history["ActivityDate"],
        history["Calories"],
        marker="o",
        linewidth=1.8,
        markersize=4
    )

    ax.set_title(
        "Історія добових енергетичних витрат"
    )

    ax.set_xlabel(
        "Дата"
    )

    ax.set_ylabel(
        "Calories, ккал"
    )

    ax.grid(
        True,
        alpha=0.25
    )

    fig.autofmt_xdate()

    plt.tight_layout()

    output = widgets.Output()

    with output:

        display(fig)

    plt.close(fig)

    return output


# ------------------------------------------------------------
# HISTORY SUMMARY
# ------------------------------------------------------------

def render_history_summary(
    user_id,
    activity_date,
    days=30
):
    """
    Коротке статистичне резюме історії.
    """

    history = prepare_history_data(
        user_id,
        activity_date,
        days=days
    )

    if history.empty:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                Немає даних для формування резюме.
            </div>
            """
        )

    calories = history["Calories"]

    current = float(
        calories.iloc[-1]
    )

    mean_value = float(
        calories.mean()
    )

    min_value = float(
        calories.min()
    )

    max_value = float(
        calories.max()
    )

    return widgets.HTML(
        value=f"""
        <div class="ui-metric-grid">

            <div class="ui-metric">

                <div class="ui-metric-label">
                    Поточне значення
                </div>

                <div class="ui-metric-value">
                    {format_kcal(current)}
                </div>

            </div>


            <div class="ui-metric">

                <div class="ui-metric-label">
                    Середнє за період
                </div>

                <div class="ui-metric-value">
                    {format_kcal(mean_value)}
                </div>

            </div>


            <div class="ui-metric">

                <div class="ui-metric-label">
                    Мінімум
                </div>

                <div class="ui-metric-value">
                    {format_kcal(min_value)}
                </div>

            </div>


            <div class="ui-metric">

                <div class="ui-metric-label">
                    Максимум
                </div>

                <div class="ui-metric-value">
                    {format_kcal(max_value)}
                </div>

            </div>

        </div>
        """
    )


# ------------------------------------------------------------
# COMPLETE HISTORY BLOCK
# ------------------------------------------------------------

def render_history(
    user_id,
    activity_date,
    days=30
):
    """
    Повний блок історії користувача.
    """

    output = widgets.Output()

    with output:

        display(
            widgets.HTML(
                value="""
                <div class="ui-card">

                    <div class="ui-card-title">
                        Історія користувача
                    </div>

                    <div class="ui-card-note">
                        Дані Calories за останні
                        доступні дні.
                    </div>

                </div>
                """
            )
        )

        display(
            render_history_summary(
                user_id,
                activity_date,
                days=days
            )
        )

        display(
            render_history_chart(
                user_id,
                activity_date,
                days=days
            )
        )

    return output


print("✓ User history renderer loaded")

✓ User history renderer loaded


In [30]:
# @title
# ============================================================
# CELL 6 — SHAP RENDERER
# ============================================================


# ------------------------------------------------------------
# CALCULATE LOCAL SHAP
# ------------------------------------------------------------

def calculate_local_shap(modeling_row):
    """
    Розрахунок локальних SHAP-значень
    для конкретного прогнозу.
    """

    X = make_feature_vector(
        modeling_row
    )

    explainer = get_shap_explainer()

    if explainer is None:
        raise RuntimeError(
            "SHAP недоступний у поточному середовищі."
        )

    shap_values = explainer.shap_values(X)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(
        shap_values
    ).reshape(-1)

    if len(shap_values) != len(feature_columns):
        raise ValueError(
            "Кількість SHAP-значень не відповідає "
            "кількості ознак."
        )

    return pd.DataFrame({
        "feature": feature_columns,
        "shap_value": shap_values,
        "feature_value": [
            modeling_row[feature]
            for feature in feature_columns
        ]
    })


# ------------------------------------------------------------
# TOP SHAP FACTORS
# ------------------------------------------------------------

def get_top_shap_factors(
    modeling_row,
    top_n=5
):
    """
    Повертає найбільш впливові ознаки
    за абсолютним SHAP-значенням.
    """

    shap_df = calculate_local_shap(
        modeling_row
    )

    shap_df["abs_shap"] = (
        shap_df["shap_value"]
        .abs()
    )

    return (
        shap_df
        .sort_values(
            "abs_shap",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# FEATURE NAME FORMATTING
# ------------------------------------------------------------

def format_feature_name(name):
    """
    Перетворення технічної назви ознаки
    у більш читабельний вигляд.
    """

    name = str(name)

    replacements = {
        "_rolling_mean_": " — ковзне середнє ",
        "_rolling_std_": " — ковзне std ",
        "_lag_": " — лаг ",
    }

    for old, new in replacements.items():

        if old in name:

            parts = name.split(old)

            return (
                parts[0]
                + new
                + parts[1]
            )

    return name


# ------------------------------------------------------------
# SINGLE SHAP FACTOR
# ------------------------------------------------------------

def render_shap_factor(
    feature,
    shap_value,
    feature_value
):
    """
    Відображення одного SHAP-фактора.
    """

    direction = (
        "збільшує прогноз"
        if shap_value > 0
        else "зменшує прогноз"
    )

    sign = (
        "+"
        if shap_value > 0
        else ""
    )

    feature_name = format_feature_name(
        feature
    )

    return f"""
    <div class="ui-factor">

        <div>

            <div class="ui-factor-name">
                {safe_text(feature_name)}
            </div>

            <div class="ui-footnote">
                Значення: {safe_text(feature_value)}
                · {direction}
            </div>

        </div>

        <div class="ui-factor-value">
            {sign}{shap_value:.2f}
        </div>

    </div>
    """


# ------------------------------------------------------------
# TOP SHAP FACTORS RENDERER
# ------------------------------------------------------------

def render_top_shap_factors(
    modeling_row,
    top_n=5
):
    """
    Відображення основних SHAP-факторів.
    """

    try:

        top = get_top_shap_factors(
            modeling_row,
            top_n=top_n
        )

    except Exception as exc:

        return widgets.HTML(
            value=f"""
            <div class="ui-status ui-status-error">

                Не вдалося розрахувати SHAP:

                {safe_text(exc)}

            </div>
            """
        )

    if top.empty:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                SHAP-фактори відсутні.
            </div>
            """
        )

    factors = []

    for _, row in top.iterrows():

        factors.append(
            render_shap_factor(
                row["feature"],
                float(row["shap_value"]),
                row["feature_value"]
            )
        )

    return widgets.HTML(
        value=f"""
        <div class="ui-card">

            <div class="ui-card-title">
                Основні фактори прогнозу
            </div>

            <div class="ui-card-note">
                Локальна інтерпретація прогнозу
                методом SHAP.
            </div>

            {"".join(factors)}

        </div>
        """
    )


# ------------------------------------------------------------
# SHAP DIRECTION SUMMARY
# ------------------------------------------------------------

def render_shap_direction_summary(
    modeling_row,
    top_n=5
):
    """
    Короткий текстовий підсумок напрямку
    впливу основних SHAP-факторів.
    """

    try:

        top = get_top_shap_factors(
            modeling_row,
            top_n=top_n
        )

    except Exception:

        return widgets.HTML(
            value=""
        )

    if top.empty:

        return widgets.HTML(
            value=""
        )

    positive = int(
        (top["shap_value"] > 0).sum()
    )

    negative = int(
        (top["shap_value"] < 0).sum()
    )

    if positive > negative:

        text = (
            "Серед основних факторів переважають "
            "ознаки, що підвищують прогноз."
        )

    elif negative > positive:

        text = (
            "Серед основних факторів переважають "
            "ознаки, що знижують прогноз."
        )

    else:

        text = (
            "Основні фактори мають змішаний "
            "напрямок впливу на прогноз."
        )

    return widgets.HTML(
        value=f"""
        <div class="ui-status">
            {text}
        </div>
        """
    )


# ------------------------------------------------------------
# COMPLETE SHAP SECTION
# ------------------------------------------------------------

def render_shap_section(
    result,
    top_n=5
):
    """
    Повний SHAP-блок dashboard.
    """

    if result is None:

        return widgets.HTML(
            value=""
        )

    modeling_row = result.get(
        "modeling_row"
    )

    if modeling_row is None:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                Дані для SHAP-аналізу відсутні.
            </div>
            """
        )

    output = widgets.Output()

    with output:

        display(
            render_top_shap_factors(
                modeling_row,
                top_n=top_n
            )
        )

        display(
            render_shap_direction_summary(
                modeling_row,
                top_n=top_n
            )
        )

    return output


print("✓ SHAP renderer loaded")

✓ SHAP renderer loaded


In [31]:
# @title
# ============================================================
# CELL 7 — DEMO & VALIDATION
# ============================================================


# ------------------------------------------------------------
# PREPARE VALIDATION RESULT
# ------------------------------------------------------------

def prepare_validation_result(result):
    """
    Підготовка даних для validation-режиму.
    """

    if result is None:
        return None

    actual = result.get(
        "actual",
        np.nan
    )

    predicted = result.get(
        "predicted",
        np.nan
    )

    if pd.isna(actual) or pd.isna(predicted):
        return None

    absolute_error = abs(
        predicted - actual
    )

    signed_error = (
        predicted - actual
    )

    if actual != 0:

        error_percent = (
            absolute_error
            / abs(actual)
            * 100
        )

    else:

        error_percent = np.nan

    return {
        "predicted": predicted,
        "actual": actual,
        "absolute_error": absolute_error,
        "signed_error": signed_error,
        "error_percent": error_percent,
    }


# ------------------------------------------------------------
# VALIDATION STATUS
# ------------------------------------------------------------

def get_validation_status(result):
    """
    Визначення статусу validation-результату.
    """

    validation = prepare_validation_result(
        result
    )

    if validation is None:

        return (
            "Немає даних для валідації."
        )

    error_percent = validation[
        "error_percent"
    ]

    if pd.isna(error_percent):

        return (
            "Фактичне значення дорівнює нулю, "
            "тому відносну похибку не розраховано."
        )

    if error_percent <= 10:

        return (
            "Прогноз має низьку відносну похибку "
            "на цьому прикладі."
        )

    if error_percent <= 20:

        return (
            "Прогноз має помірну відносну похибку "
            "на цьому прикладі."
        )

    return (
        "Для цього прикладу спостерігається "
        "помітна похибка прогнозування."
    )


# ------------------------------------------------------------
# VALIDATION CARD
# ------------------------------------------------------------

def render_validation_card(result):
    """
    Картка з результатами validation.
    """

    validation = prepare_validation_result(
        result
    )

    if validation is None:

        return widgets.HTML(
            value="""
            <div class="ui-status">
                Дані для валідації відсутні.
            </div>
            """
        )

    predicted = validation["predicted"]
    actual = validation["actual"]
    absolute_error = validation["absolute_error"]
    signed_error = validation["signed_error"]
    error_percent = validation["error_percent"]

    return widgets.HTML(
        value=f"""
        <div class="ui-card">

            <div class="ui-card-title">
                Перевірка прогнозу
            </div>

            <div class="ui-metric-grid">

                <div class="ui-metric">

                    <div class="ui-metric-label">
                        Прогноз моделі
                    </div>

                    <div class="ui-metric-value">
                        {format_kcal(predicted)}
                    </div>

                </div>


                <div class="ui-metric">

                    <div class="ui-metric-label">
                        Фактичне значення
                    </div>

                    <div class="ui-metric-value">
                        {format_kcal(actual)}
                    </div>

                </div>


                <div class="ui-metric">

                    <div class="ui-metric-label">
                        Абсолютна похибка
                    </div>

                    <div class="ui-metric-value">
                        {format_kcal(absolute_error)}
                    </div>

                </div>


                <div class="ui-metric">

                    <div class="ui-metric-label">
                        Відносна похибка
                    </div>

                    <div class="ui-metric-value">
                        {format_percent(error_percent)}
                    </div>

                </div>

            </div>

            <div class="ui-card-note">

                Знак похибки:
                <b>{format_signed_kcal(signed_error)}</b>

                — різниця між прогнозом
                та фактичним значенням.

            </div>

        </div>
        """
    )


# ------------------------------------------------------------
# VALIDATION VS BASELINES
# ------------------------------------------------------------

def render_validation_baseline_result(result):
    """
    Порівняння помилки моделі з baseline
    у validation-режимі.
    """

    if result is None:

        return widgets.HTML(
            value=""
        )

    actual = result.get(
        "actual",
        np.nan
    )

    predicted = result.get(
        "predicted",
        np.nan
    )

    baseline_7day = result.get(
        "baseline_7day",
        np.nan
    )

    baseline_user = result.get(
        "baseline_user",
        np.nan
    )

    if (
        pd.isna(actual)
        or pd.isna(predicted)
    ):

        return widgets.HTML(
            value=""
        )

    model_error = abs(
        predicted - actual
    )

    cards = [
        f"""
        <div class="ui-metric">

            <div class="ui-metric-label">
                Помилка Random Forest
            </div>

            <div class="ui-metric-value">
                {format_kcal(model_error)}
            </div>

        </div>
        """
    ]

    if not pd.isna(baseline_7day):

        baseline_error = abs(
            baseline_7day - actual
        )

        cards.append(
            f"""
            <div class="ui-metric">

                <div class="ui-metric-label">
                    Помилка 7-Day Mean
                </div>

                <div class="ui-metric-value">
                    {format_kcal(baseline_error)}
                </div>

            </div>
            """
        )

    if not pd.isna(baseline_user):

        baseline_error = abs(
            baseline_user - actual
        )

        cards.append(
            f"""
            <div class="ui-metric">

                <div class="ui-metric-label">
                    Помилка User Mean
                </div>

                <div class="ui-metric-value">
                    {format_kcal(baseline_error)}
                </div>

            </div>
            """
        )

    return widgets.HTML(
        value=f"""
        <div class="ui-card">

            <div class="ui-card-title">
                Помилка моделі та baseline
            </div>

            <div class="ui-metric-grid">
                {"".join(cards)}
            </div>

        </div>
        """
    )


print("✓ Demo & validation renderer loaded")

✓ Demo & validation renderer loaded


In [32]:
# @title
# ============================================================
# CELL 8 — FINAL DASHBOARD ASSEMBLY
# ============================================================


# ------------------------------------------------------------
# DASHBOARD HEADER
# ------------------------------------------------------------

def render_dashboard_header():
    """
    Заголовок фінального dashboard.
    """

    return widgets.HTML(
        value="""
        <div class="ui-header">

            <div class="ui-title">
                Прогноз добових енергетичних витрат
            </div>

            <div class="ui-subtitle">
                Інтелектуальна система прогнозування
                на основі даних фітнес-трекера
            </div>

        </div>
        """
    )


# ------------------------------------------------------------
# DASHBOARD CONTROLS
# ------------------------------------------------------------

def render_dashboard_controls():
    """
    Панель керування dashboard.
    """

    controls = widgets.VBox(
        [
            widgets.HBox(
                [
                    user_dropdown,
                    mode_dropdown,
                    date_dropdown
                ],
                layout=widgets.Layout(
                    flex_wrap="wrap",
                    gap="10px"
                )
            ),

            forecast_button

        ],
        layout=widgets.Layout(
            gap="12px"
        )
    )

    return widgets.VBox(
        [
            widgets.HTML(
                value="""
                <div class="ui-section-title">
                    Параметри прогнозування
                </div>
                """
            ),

            controls
        ]
    )


# ------------------------------------------------------------
# EMPTY DASHBOARD STATE
# ------------------------------------------------------------

def render_empty_dashboard_state():
    """
    Початковий стан dashboard.
    """

    return widgets.HTML(
        value="""
        <div class="ui-card">

            <div class="ui-card-title">
                Система готова до роботи
            </div>

            <div class="ui-status">
                Оберіть користувача, режим та дату,
                після чого натисніть
                <b>«Сформувати прогноз»</b>.
            </div>

        </div>
        """
    )


# ------------------------------------------------------------
# MODEL VS BASELINE
# ------------------------------------------------------------

def render_baseline_comparison(result):
    """
    Порівняння прогнозу Random Forest
    з базовими стратегіями.
    """

    if result is None:

        return widgets.HTML(
            value=""
        )

    predicted = result.get(
        "predicted",
        np.nan
    )

    baseline_7day = result.get(
        "baseline_7day",
        np.nan
    )

    baseline_user = result.get(
        "baseline_user",
        np.nan
    )

    if pd.isna(predicted):

        return widgets.HTML(
            value=""
        )

    cards = [
        f"""
        <div class="ui-metric">

            <div class="ui-metric-label">
                Random Forest
            </div>

            <div class="ui-metric-value">
                {format_kcal(predicted)}
            </div>

            <div class="ui-metric-small">
                прогноз моделі
            </div>

        </div>
        """
    ]

    if not pd.isna(baseline_7day):

        cards.append(
            f"""
            <div class="ui-metric">

                <div class="ui-metric-label">
                    7-Day Mean
                </div>

                <div class="ui-metric-value">
                    {format_kcal(baseline_7day)}
                </div>

                <div class="ui-metric-small">
                    базовий прогноз
                </div>

            </div>
            """
        )

    if not pd.isna(baseline_user):

        cards.append(
            f"""
            <div class="ui-metric">

                <div class="ui-metric-label">
                    User Mean
                </div>

                <div class="ui-metric-value">
                    {format_kcal(baseline_user)}
                </div>

                <div class="ui-metric-small">
                    індивідуальний baseline
                </div>

            </div>
            """
        )

    return widgets.HTML(
        value=f"""
        <div class="ui-card">

            <div class="ui-card-title">
                Модель та базові стратегії
            </div>

            <div class="ui-metric-grid">
                {"".join(cards)}
            </div>

        </div>
        """
    )


# ------------------------------------------------------------
# MODEL ADVANTAGE
# ------------------------------------------------------------

def render_model_advantage(result):
    """
    Коротке порівняння прогнозу моделі
    з базовими стратегіями.
    """

    if result is None:

        return widgets.HTML(
            value=""
        )

    predicted = result.get(
        "predicted",
        np.nan
    )

    baseline_7day = result.get(
        "baseline_7day",
        np.nan
    )

    baseline_user = result.get(
        "baseline_user",
        np.nan
    )

    if pd.isna(predicted):

        return widgets.HTML(
            value=""
        )

    comparisons = []

    if not pd.isna(baseline_7day):

        diff = predicted - baseline_7day

        comparisons.append(
            "7-Day Mean: "
            + format_signed_kcal(diff)
        )

    if not pd.isna(baseline_user):

        diff = predicted - baseline_user

        comparisons.append(
            "User Mean: "
            + format_signed_kcal(diff)
        )

    if not comparisons:

        return widgets.HTML(
            value=""
        )

    comparison_text = " · ".join(
        comparisons
    )

    return widgets.HTML(
        value=f"""
        <div class="ui-status">

            <b>Порівняння прогнозів:</b>

            {safe_text(comparison_text)}

        </div>
        """
    )


# ------------------------------------------------------------
# DASHBOARD RESULT
# ------------------------------------------------------------

def render_dashboard_result(
    result,
    mode
):
    """
    Повне відображення результату прогнозування.
    """

    output = widgets.Output()

    with output:

        # ----------------------------------------------------
        # FORECAST
        # ----------------------------------------------------

        display(
            render_forecast_card(
                result
            )
        )

        display(
            render_forecast_status(
                result,
                mode=mode
            )
        )


        # ----------------------------------------------------
        # MODEL VS BASELINE
        # ----------------------------------------------------

        display(
            render_baseline_comparison(
                result
            )
        )

        display(
            render_model_advantage(
                result
            )
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        if mode == "validation":

            display(
                render_validation_card(
                    result
                )
            )

            display(
                render_validation_baseline_result(
                    result
                )
            )


        # ----------------------------------------------------
        # SHAP
        # ----------------------------------------------------

        display(
            render_shap_section(
                result,
                top_n=SHAP_TOP_N
            )
        )


        # ----------------------------------------------------
        # HISTORY
        # ----------------------------------------------------

        display(
            render_history(
                result["user_id"],
                result["activity_date"],
                days=HISTORY_DAYS
            )
        )

    return output


# ------------------------------------------------------------
# DASHBOARD ERROR
# ------------------------------------------------------------

def render_dashboard_error(error):
    """
    Відображення зрозумілої помилки.
    """

    return widgets.HTML(
        value=f"""
        <div class="ui-error">

            <b>Не вдалося сформувати прогноз.</b>

            <br><br>

            {safe_text(error)}

        </div>
        """
    )


# ------------------------------------------------------------
# FORECAST BUTTON CALLBACK
# ------------------------------------------------------------

def on_forecast_button_click(
    button
):
    """
    Основний callback кнопки прогнозування.
    """

    dashboard_output.clear_output()

    user_id = user_dropdown.value
    mode = mode_dropdown.value
    activity_date = date_dropdown.value

    if user_id is None:

        with dashboard_output:

            display(
                render_dashboard_error(
                    "Не вибрано користувача."
                )
            )

        return

    if activity_date is None:

        with dashboard_output:

            display(
                render_dashboard_error(
                    "Не вибрано дату."
                )
            )

        return

    try:

        # ----------------------------------------------------
        # REAL PREDICTION
        # ----------------------------------------------------

        if mode == "real":

            result = build_prediction_result(
                user_id,
                activity_date,
                validation=False
            )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        elif mode == "validation":

            result = build_prediction_result(
                user_id,
                activity_date,
                validation=True
            )


        # ----------------------------------------------------
        # DEMO
        # ----------------------------------------------------

        elif mode == "demo":

            result = build_prediction_result(
                user_id,
                activity_date,
                validation=False
            )


        else:

            raise ValueError(
                f"Невідомий режим: {mode}"
            )


        with dashboard_output:

            display(
                render_dashboard_result(
                    result,
                    mode
                )
            )


    except Exception as exc:

        with dashboard_output:

            display(
                render_dashboard_error(
                    exc
                )
            )


# ------------------------------------------------------------
# CONNECT BUTTON
# ------------------------------------------------------------

try:
    forecast_button.on_click(
        on_forecast_button_click
    )
except Exception:
    pass


print("✓ Final dashboard assembly loaded")

✓ Final dashboard assembly loaded


In [33]:
# @title
# ============================================================
# CELL 9 — FINAL UI COMPOSITION
# ============================================================


def build_dashboard():
    """
    Формує основний контейнер dashboard.
    """

    header = render_dashboard_header()

    controls_title = widgets.HTML(
        value="""
        <div class="ui-section-title">
            Параметри прогнозування
        </div>
        """
    )

    controls_box = widgets.VBox(
        [
            widgets.HBox(
                [
                    user_dropdown,
                    mode_dropdown,
                    date_dropdown
                ],
                layout=widgets.Layout(
                    flex_flow="row wrap",
                    gap="12px"
                )
            ),

            forecast_button

        ],
        layout=widgets.Layout(
            gap="14px"
        )
    )

    controls_card = widgets.VBox(
        [
            controls_title,
            controls_box
        ],
        layout=widgets.Layout(
            gap="10px"
        )
    )

    dashboard_output.clear_output()

    with dashboard_output:

        display(
            render_empty_dashboard_state()
        )

    return widgets.VBox(
        [
            header,
            controls_card,
            dashboard_output
        ],
        layout=widgets.Layout(
            width="100%",
            gap="18px"
        )
    )


print("✓ UI composition loaded")

✓ UI composition loaded


In [39]:
# @title
# ============================================================
# CELL 11 — FINAL UI POLISH
# ============================================================


# ------------------------------------------------------------
# FINAL CSS
# ------------------------------------------------------------

FINAL_UI_CSS = r"""
<style>

.ui-header {
    padding: 24px 28px;
    margin-bottom: 4px;
    border-radius: 14px;
    background: linear-gradient(
        135deg,
        #172033 0%,
        #202b42 100%
    );
    border: 1px solid #34415c;
}

.ui-title {
    font-size: 28px;
    font-weight: 700;
    line-height: 1.25;
    margin-bottom: 8px;
    color: #ffffff;
}

.ui-subtitle {
    font-size: 15px;
    line-height: 1.5;
    color: #b8c2d6;
}

.ui-section-title {
    font-size: 18px;
    font-weight: 600;
    margin: 4px 0 10px 0;
    color: #e8edf7;
}

.ui-card {
    padding: 20px;
    margin-top: 10px;
    border-radius: 14px;
    background: #20242b;
    border: 1px solid #363d49;
}

.ui-card-title {
    font-size: 18px;
    font-weight: 600;
    margin-bottom: 8px;
    color: #f2f4f8;
}

.ui-card-note {
    font-size: 13px;
    color: #aeb7c7;
}

.ui-status {
    padding: 12px 15px;
    margin-top: 12px;
    border-radius: 9px;
    background: #292f38;
    border: 1px solid #414957;
    color: #d7dce5;
}

.ui-status-success {
    border-color: #3f805e;
    background: #202f28;
}

.ui-status-warning {
    border-color: #8a7138;
    background: #302b20;
}

.ui-status-error {
    border-color: #8a4b4b;
    background: #302222;
}

.ui-forecast {
    padding: 24px;
    margin-top: 14px;
    border-radius: 14px;
    background: #202a38;
    border: 1px solid #3c4d67;
    text-align: center;
}

.ui-forecast-label {
    font-size: 14px;
    color: #aebbd0;
    margin-bottom: 6px;
}

.ui-forecast-value {
    font-size: 40px;
    font-weight: 700;
    line-height: 1.1;
    color: #ffffff;
}

.ui-forecast-unit {
    font-size: 15px;
    color: #b9c3d3;
}

.ui-metric-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
    margin-top: 14px;
}

.ui-metric {
    padding: 15px;
    border-radius: 10px;
    background: #272c34;
    border: 1px solid #383f4a;
}

.ui-metric-label {
    font-size: 12px;
    color: #aab3c2;
    margin-bottom: 5px;
}

.ui-metric-value {
    font-size: 20px;
    font-weight: 600;
    color: #f1f3f7;
}

.ui-factor {
    padding: 13px 15px;
    margin-top: 8px;
    border-radius: 9px;
    background: #252b34;
    border: 1px solid #383f4a;
}

.ui-two-columns {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
    margin-top: 16px;
}

.ui-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
}

.ui-table th,
.ui-table td {
    padding: 9px 10px;
    border-bottom: 1px solid #383f4a;
    text-align: left;
}

.ui-table th {
    color: #b7c0cf;
    font-weight: 600;
}

.ui-table td {
    color: #e4e7ec;
}

.ui-error {
    padding: 16px;
    margin-top: 14px;
    border-radius: 10px;
    background: #302222;
    border: 1px solid #8a4b4b;
    color: #f0caca;
}

.ui-footnote {
    margin-top: 10px;
    font-size: 12px;
    color: #8f98a8;
}

.ui-mode {
    padding: 10px 14px;
    border-radius: 8px;
    background: #292f38;
    color: #d9dee8;
}

.widget-label {
    color: #dce2ec !important;
}

.jp-OutputArea-output {
    overflow-x: auto;
}

@media (max-width: 850px) {

    .ui-title {
        font-size: 23px;
    }

    .ui-metric-grid {
        grid-template-columns: 1fr;
    }

    .ui-two-columns {
        grid-template-columns: 1fr;
    }

}

</style>
"""


final_style_widget = widgets.HTML(
    value=FINAL_UI_CSS
)


# ------------------------------------------------------------
# BUILD FINAL DASHBOARD
# ------------------------------------------------------------

def build_final_dashboard():
    """
    Формує фінальну сторінку разом із CSS.
    """

    header = render_dashboard_header()

    controls_title = widgets.HTML(
        value="""
        <div class="ui-section-title">
            Параметри прогнозування
        </div>
        """
    )

    controls_box = widgets.VBox(
        [
            widgets.HBox(
                [
                    user_dropdown,
                    mode_dropdown,
                    date_dropdown
                ],
                layout=widgets.Layout(
                    flex_flow="row wrap",
                    gap="12px",
                    width="100%"
                )
            ),

            forecast_button

        ],
        layout=widgets.Layout(
            gap="14px",
            width="100%"
        )
    )

    controls_card = widgets.VBox(
        [
            controls_title,
            controls_box
        ],
        layout=widgets.Layout(
            gap="8px",
            width="100%"
        )
    )

    dashboard_output.clear_output()

    with dashboard_output:

        display(
            render_empty_dashboard_state()
        )

    return widgets.VBox(
        [
            final_style_widget,
            header,
            controls_card,
            dashboard_output
        ],
        layout=widgets.Layout(
            width="100%",
            max_width="1200px",
            margin="0 auto",
            padding="20px",
            gap="18px"
        )
    )


# ============================================================
# VOILÀ FINAL DISPLAY
# ============================================================

final_dashboard = build_final_dashboard()

display(
    widgets.VBox(
        [
            final_dashboard
        ]
    )
)